In [122]:
import pandas as pd
import numpy as np

# Exploratory Data Analysis

In [123]:
df = pd.read_csv(r'D:\Data Analysis\data analysis intern\Task6\Data_Files\Raw_Files\Weather_Data.csv')

In [124]:
df

,Date,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2/1/2008,19.5,22.4,15.6,6.2,0.0,W,41,S,SSW,...,92,84,1017.6,1017.4,8,8,20.7,20.9,Yes,Yes
1,2/2/2008,19.5,25.6,6.0,3.4,2.7,W,41,W,E,...,83,73,1017.9,1016.4,7,7,22.4,24.8,Yes,Yes
2,2/3/2008,21.6,24.5,6.6,2.4,0.1,W,41,ESE,ESE,...,88,86,1016.7,1015.6,7,8,23.5,23.0,Yes,Yes
3,2/4/2008,20.2,22.8,18.8,2.2,0.0,W,41,NNE,E,...,83,90,1014.2,1011.8,8,8,21.4,20.9,Yes,Yes
4,2/5/2008,19.7,25.7,77.4,4.8,0.0,W,41,NNE,W,...,88,74,1008.3,1004.8,8,8,22.5,25.5,Yes,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3266,6/21/2017,8.6,19.6,0.0,2.0,7.8,SSE,37,W,SSE,...,73,52,1025.9,1025.3,2,2,10.5,17.9,No,No
3267,6/22/2017,9.3,19.2,0.0,2.0,9.2,W,30,W,ESE,...,78,53,1028.5,1024.6,2,2,11.0,18.7,No,No
3268,6/23/2017,9.4,17.7,0.0,2.4,2.7,W,24,WNW,N,...,85,56,1020.8,1015.0,6,6,10.2,17.3,No,No
3269,6/24/2017,10.1,19.3,0.0,1.4,9.3,W,43,W,W,...,56,35,1017.3,1015.1,5,2,12.4,19.0,No,No


In [125]:
df.info()
# 1- change date from object to date

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3271 entries, 0 to 3270
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           3271 non-null   object 
 1   MinTemp        3271 non-null   float64
 2   MaxTemp        3271 non-null   float64
 3   Rainfall       3271 non-null   float64
 4   Evaporation    3271 non-null   float64
 5   Sunshine       3271 non-null   float64
 6   WindGustDir    3271 non-null   object 
 7   WindGustSpeed  3271 non-null   int64  
 8   WindDir9am     3271 non-null   object 
 9   WindDir3pm     3271 non-null   object 
 10  WindSpeed9am   3271 non-null   int64  
 11  WindSpeed3pm   3271 non-null   int64  
 12  Humidity9am    3271 non-null   int64  
 13  Humidity3pm    3271 non-null   int64  
 14  Pressure9am    3271 non-null   float64
 15  Pressure3pm    3271 non-null   float64
 16  Cloud9am       3271 non-null   int64  
 17  Cloud3pm       3271 non-null   int64  
 18  Temp9am 

In [126]:
# Convert 'Date' to datetime and sort chronologically
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
df = df.sort_values("Date").reset_index(drop=True)

print(f"Date range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")

Date range: 2008-02-01 to 2017-06-25


In [127]:
df.isnull().sum()

Date             0
MinTemp          0
MaxTemp          0
Rainfall         0
Evaporation      0
Sunshine         0
WindGustDir      0
WindGustSpeed    0
WindDir9am       0
WindDir3pm       0
WindSpeed9am     0
WindSpeed3pm     0
Humidity9am      0
Humidity3pm      0
Pressure9am      0
Pressure3pm      0
Cloud9am         0
Cloud3pm         0
Temp9am          0
Temp3pm          0
RainToday        0
RainTomorrow     0
dtype: int64

In [128]:
df.duplicated().sum()

np.int64(0)

In [129]:
# Perform domain boundary and sanity checks across key variables
min_gt_max = (df["MinTemp"] > df["MaxTemp"]).sum()
hum_out_of_bounds = ((df["Humidity9am"] < 0) | (df["Humidity9am"] > 100) | (df["Humidity3pm"] < 0) | (df["Humidity3pm"] > 100)).sum()
temp3pm_gt_max = (df["Temp3pm"] > df["MaxTemp"]).sum()
negative_wind = ((df["WindGustSpeed"] < 0) | (df["WindSpeed9am"] < 0) | (df["WindSpeed3pm"] < 0)).sum()

print("--- Logic Checks Results ---")
print(f"1. MinTemp > MaxTemp violations:          {min_gt_max}")
print(f"2. Humidity outside [0, 100]% violations: {hum_out_of_bounds}")
print(f"3. Temp3pm > MaxTemp violations:          {temp3pm_gt_max}")
print(f"4. Negative wind speed violations:        {negative_wind}")

--- Logic Checks Results ---
1. MinTemp > MaxTemp violations:          0
2. Humidity outside [0, 100]% violations: 0
3. Temp3pm > MaxTemp violations:          0
4. Negative wind speed violations:        0


In [130]:
# Compare RainToday binary indicator against Rainfall threshold (>= 1.0 mm)
rainfall_rule_yes = df["Rainfall"] >= 1.0
raintoday_yes = df["RainToday"] == "Yes"

mismatches = df[rainfall_rule_yes != raintoday_yes]

print(f"Total RainToday vs. Rainfall mismatches: {len(mismatches)}")

# Detailed inspection of the mismatch rows
print("\n--- Summary Breakdown of Mismatches ---")
print(mismatches[["Date", "Rainfall", "RainToday"]].value_counts("Rainfall"))

print("\n--- Sample Mismatch Rows ---")
display(mismatches[["Date", "Rainfall", "RainToday"]].head(10))

Total RainToday vs. Rainfall mismatches: 37

--- Summary Breakdown of Mismatches ---
Rainfall
1.0    37
Name: count, dtype: int64

--- Sample Mismatch Rows ---


,Date,Rainfall,RainToday
49,2008-03-21,1.0,No
117,2008-05-29,1.0,No
354,2009-01-22,1.0,No
623,2009-10-19,1.0,No
660,2009-11-27,1.0,No
695,2010-01-04,1.0,No
827,2010-05-17,1.0,No
830,2010-05-20,1.0,No
864,2010-06-26,1.0,No
881,2010-07-14,1.0,No


In [131]:
# Align each row with the next calendar day's RainToday
df["Next_RainToday"] = df["RainToday"].shift(-1)
df["Next_Date"] = df["Date"].shift(-1)

# Flag pairs that are strictly consecutive calendar days (+1 day)
df["Is_Consecutive"] = (df["Next_Date"] - df["Date"]) == pd.Timedelta(days=1)

consecutive_df = df[df["Is_Consecutive"]].copy()
matches = consecutive_df["RainTomorrow"] == consecutive_df["Next_RainToday"]

match_pct = matches.mean() * 100
mismatch_count = (~matches).sum()

print("--- RainTomorrow Integrity Verification ---")
print(f"Total consecutive day pairs evaluated: {len(consecutive_df)}")
print(f"Exact matches count:                    {matches.sum()}")
print(f"Mismatches count on consecutive days:   {mismatch_count}")
print(f"Match percentage:                      {match_pct:.2f}%")

--- RainTomorrow Integrity Verification ---
Total consecutive day pairs evaluated: 3225
Exact matches count:                    3225
Mismatches count on consecutive days:   0
Match percentage:                      100.00%


# Data Cleaning and Feature Engineering for Analysis

In [132]:
# Flag Date Gaps (DO NOT interpolate/fill missing dates; note that averages use available days)
df["Date_Diff_Days"] = df["Date"].diff().dt.days
df["Is_After_Gap"] = (df["Date_Diff_Days"] > 1).astype(int)

# Flag Extreme Outliers (DO NOT delete real extreme weather events like 45.8°C or 119.4 mm rain)
# Outlier criteria: MaxTemp >= 40.0°C OR Rainfall >= 50.0 mm OR WindGustSpeed >= 80 km/h
df["is_outlier"] = ((df["MaxTemp"] >= 40.0) | (df["Rainfall"] >= 50.0) | (df["WindGustSpeed"] >= 80)).astype(int)

print(f"Total Date Gaps Flagged: {df['Is_After_Gap'].sum()}")
print(f"Total Extreme Outlier Days Flagged: {df['is_outlier'].sum()}")

Total Date Gaps Flagged: 45
Total Extreme Outlier Days Flagged: 61


In [133]:
# 1. Temporal Features (Powering Dashboard Slicers)
df["Year"] = df["Date"].dt.year
df["Quarter"] = df["Date"].dt.quarter
df["Month"] = df["Date"].dt.month
df["MonthName"] = df["Date"].dt.strftime("%b")
df["YearMonth"] = df["Date"].dt.strftime("%Y-%m")


In [134]:
# 2. Southern Hemisphere Season Mapping
# Assumption: Peak temperatures occur in Jan-Feb and coldest in July (Dec-Feb = Summer)
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

df["Season"] = df["Month"].apply(get_season)


In [135]:
# 3. Temperature Derived Features
df["AvgTemp"] = round((df["MinTemp"] + df["MaxTemp"]) / 2.0,2)
df["TempRange"] = round(df["MaxTemp"] - df["MinTemp"], 2)
df["HeatDay"] = (df["MaxTemp"] >= 35.0).astype(int)

print("Sample Temperature & Seasonality Features:")
display(df[["Date", "Year", "Season", "AvgTemp", "TempRange", "HeatDay"]].head())

Sample Temperature & Seasonality Features:


,Date,Year,Season,AvgTemp,TempRange,HeatDay
0,2008-02-01,2008,Winter,20.95,2.9,0
1,2008-02-02,2008,Winter,22.55,6.1,0
2,2008-02-03,2008,Winter,23.05,2.9,0
3,2008-02-04,2008,Winter,21.50,2.6,0
4,2008-02-05,2008,Winter,22.70,6.0,0


In [136]:
# 1. Rainfall Indicators & Binned Categories
df["RainDay_1mm"] = (df["Rainfall"] >= 1.0).astype(int)

def get_rain_category(r):
    if r == 0.0:
        return "None"
    elif r < 2.5:
        return "Light"
    elif r < 10.0:
        return "Moderate"
    elif r <= 50.0:
        return "Heavy"
    else:
        return "Extreme"

df["RainCategory"] = df["Rainfall"].apply(get_rain_category)


In [137]:
# 2. Wind Speed Bands
def get_wind_category(speed):
    if speed < 30:
        return "Light (<30 km/h)"
    elif speed < 45:
        return "Moderate (30-45 km/h)"
    elif speed <= 60:
        return "Strong (45-60 km/h)"
    else:
        return "Gale (>60 km/h)"

df["WindCategory"] = df["WindGustSpeed"].apply(get_wind_category)


In [138]:
# 3. Compass Direction Grouped into Quadrants (N / E / S / W)
def get_quadrant(dir_str):
    if pd.isna(dir_str):
        return "Unknown"
    d = str(dir_str).upper()
    if d in ["N", "NNE", "NNW", "NE", "NW"]:
        return "North"
    elif d in ["E", "ENE", "ESE", "SE"]:
        return "East"
    elif d in ["S", "SSE", "SSW", "SW"]:
        return "South"
    elif d in ["W", "WSW", "WNW"]:
        return "West"
    return "Other"

df["WindGustDir_Quadrant"] = df["WindGustDir"].apply(get_quadrant)

In [139]:
df

,Date,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,MonthName,YearMonth,Season,AvgTemp,TempRange,HeatDay,RainDay_1mm,RainCategory,WindCategory,WindGustDir_Quadrant
0,2008-02-01,19.5,22.4,15.6,6.2,0.0,W,41,S,SSW,...,Feb,2008-02,Winter,20.95,2.9,0,1,Heavy,Moderate (30-45 km/h),West
1,2008-02-02,19.5,25.6,6.0,3.4,2.7,W,41,W,E,...,Feb,2008-02,Winter,22.55,6.1,0,1,Moderate,Moderate (30-45 km/h),West
2,2008-02-03,21.6,24.5,6.6,2.4,0.1,W,41,ESE,ESE,...,Feb,2008-02,Winter,23.05,2.9,0,1,Moderate,Moderate (30-45 km/h),West
3,2008-02-04,20.2,22.8,18.8,2.2,0.0,W,41,NNE,E,...,Feb,2008-02,Winter,21.50,2.6,0,1,Heavy,Moderate (30-45 km/h),West
4,2008-02-05,19.7,25.7,77.4,4.8,0.0,W,41,NNE,W,...,Feb,2008-02,Winter,22.70,6.0,0,1,Extreme,Moderate (30-45 km/h),West
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3266,2017-06-21,8.6,19.6,0.0,2.0,7.8,SSE,37,W,SSE,...,Jun,2017-06,Summer,14.10,11.0,0,0,None,Moderate (30-45 km/h),South
3267,2017-06-22,9.3,19.2,0.0,2.0,9.2,W,30,W,ESE,...,Jun,2017-06,Summer,14.25,9.9,0,0,None,Moderate (30-45 km/h),West
3268,2017-06-23,9.4,17.7,0.0,2.4,2.7,W,24,WNW,N,...,Jun,2017-06,Summer,13.55,8.3,0,0,None,Light (<30 km/h),West
3269,2017-06-24,10.1,19.3,0.0,1.4,9.3,W,43,W,W,...,Jun,2017-06,Summer,14.70,9.2,0,0,None,Moderate (30-45 km/h),West


In [140]:
df.to_csv(r'D:\Data Analysis\data analysis intern\Task6\Data_Files\Cleaned_Files\cleaned_data.csv')